# Chronos-2 Zero-Shot Baseline (Close Price Target)

## Overview
- **Input**: `data/processed/v2/features_train.csv`, `data/processed/v2/features_test.csv`
- **Output**: `data/results/chronos_results.csv`, `data/results/chronos_summary.csv`

## Strategy
- Feed raw Close price series as context via `predict_df()`
- Calendar covariates in `future_df`: day_of_week, quarter, month
- `cross_learning=True`: all 15 tickers predicted together per step
- Rolling 1-step: for each test day, predict next-day Close price
- Convert pred Close -> log return for metric comparison with other models

**Model**: `amazon/chronos-2`  
**Zero-shot**: No fine-tuning, weights frozen.

In [25]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path

warnings.filterwarnings("ignore")
if sys.stdout.encoding and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

In [26]:
# ========== SETUP PATHS & CONFIG ==========
ROOT_DIR   = Path.cwd().parent.parent
TRAIN_PATH = ROOT_DIR / "data" / "processed" / "v2" / "features_train.csv"
TEST_PATH  = ROOT_DIR / "data" / "processed" / "v2" / "features_test.csv"
OUT_DIR    = ROOT_DIR / "data" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_LEN = 512
MODEL_ID    = "amazon/chronos-2"
BATCH_SIZE  = 100

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"TRAIN_PATH: {TRAIN_PATH}")
print(f"TEST_PATH: {TEST_PATH}")
print(f"OUT_DIR: {OUT_DIR}")

ROOT_DIR: d:\fm-forecasting
TRAIN_PATH: d:\fm-forecasting\data\processed\v2\features_train.csv
TEST_PATH: d:\fm-forecasting\data\processed\v2\features_test.csv
OUT_DIR: d:\fm-forecasting\data\results


In [27]:
# ========== HELPER FUNCTIONS ==========
def eval_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    da   = np.mean(np.sign(y_true) == np.sign(y_pred))
    ic   = np.corrcoef(y_true, y_pred)[0, 1] if np.std(y_pred) > 0 else 0.0
    return {"MAE": mae, "RMSE": rmse, "DA": da, "IC": ic}

def add_calendar_features(df: pd.DataFrame, date_col: str = "timestamp") -> pd.DataFrame:
    """Add calendar covariates for Chronos-2."""
    df = df.copy()
    dt = pd.to_datetime(df[date_col])
    df["day_of_week"]       = dt.dt.dayofweek
    df["quarter"]           = dt.dt.quarter
    df["month"]             = dt.dt.month
    df["is_derivative_exp"] = 0
    return df

In [28]:
# ========== LOAD DATA ==========
print("Loading training and test data...")
train = pd.read_csv(TRAIN_PATH, parse_dates=["Date"])
test  = pd.read_csv(TEST_PATH,  parse_dates=["Date"])

tickers = sorted(train["ticker"].unique())
print(f"Tickers ({len(tickers)}): {tickers}")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Loading training and test data...
Tickers (15): ['ACB', 'BVH', 'CTG', 'FPT', 'GAS', 'HPG', 'KDH', 'MBB', 'MSN', 'PDR', 'SSI', 'STB', 'VCB', 'VIC', 'VNM']
Train shape: (34917, 21)
Test shape: (13140, 21)


In [29]:
# ========== LOAD CHRONOS-2 MODEL ==========
def load_model():
    try:
        from chronos import Chronos2Pipeline
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype  = torch.bfloat16 if device == "cuda" else torch.float32
        pipeline = Chronos2Pipeline.from_pretrained(
            MODEL_ID,
            device_map=device,
            torch_dtype=dtype,
        )
        print(f"Chronos-2 loaded | device={device}")
        return pipeline
    except ImportError:
        print("ERROR: chronos not installed. Run: pip install chronos-forecasting")
        raise

pipeline = load_model()

Chronos-2 loaded | device=cpu


In [30]:
# ========== PREPARE PANEL DATA ==========
train_panel = train[["ticker", "Date", "Close"]].rename(columns={"ticker": "id", "Date": "timestamp"})
test_panel  = test[["ticker", "Date", "Close", "target", "regime_hmm"]].rename(columns={"ticker": "id", "Date": "timestamp"})

df_all = pd.concat([
    train_panel,
    test_panel[["id", "timestamp", "Close"]],
], ignore_index=True)

# Resample each ticker to Business Days so Chronos can infer a stable frequency.
dfs = []
for ticker_id, group in df_all.groupby("id"):
    g = group.sort_values("timestamp").set_index("timestamp").resample("B").ffill()
    g["id"] = ticker_id
    dfs.append(g.reset_index())

all_close = pd.concat(dfs, ignore_index=True)[["id", "timestamp", "Close"]]

test_dates = sorted(test_panel["timestamp"].unique())
print(f"Test days: {len(test_dates)} | from {test_dates[0].date()} to {test_dates[-1].date()}")
print(f"Resampled panel rows: {len(all_close)}")

Test days: 876 | from 2022-07-01 to 2025-12-30
Resampled panel rows: 50205


In [31]:
# ========== ROLLING 1-STEP FORECAST HELPERS ==========
def build_context_df(all_data: pd.DataFrame, before_date: pd.Timestamp, context_len: int) -> pd.DataFrame:
    """Get context_len rows before before_date for all tickers."""
    ctx = all_data[all_data["timestamp"] < before_date].copy()
    ctx = (
        ctx.groupby("id", group_keys=False)
           .apply(lambda g: g.sort_values("timestamp").tail(context_len))
           .reset_index(drop=True)
    )
    return add_calendar_features(ctx)

pred_store = {}  # {(ticker, date): pred_return}
print("Helper functions ready")

Helper functions ready


In [32]:
# ========== ROLLING 1-STEP INFERENCE ==========
print("Running rolling inference...")

for step, test_date in enumerate(test_dates):
    if (step + 1) % 50 == 0 or step == 0:
        print(f"  Step {step+1}/{len(test_dates)} | date={test_date.date()}")

    ctx_df = build_context_df(all_close, test_date, CONTEXT_LEN)
    if ctx_df.empty:
        continue

    future_df = pd.DataFrame([{"id": ticker, "timestamp": test_date} for ticker in tickers])

    try:
        forecast_df = pipeline.predict_df(
            df=ctx_df,
            future_df=future_df[["id", "timestamp"]],
            prediction_length=1,
            quantile_levels=[0.5],
            id_column="id",
            timestamp_column="timestamp",
            target="Close",
            cross_learning=True,
            batch_size=BATCH_SIZE,
        )

        for _, row in forecast_df.iterrows():
            ticker     = row["id"]
            pred_close = float(row["predictions"])

            close_current_ser = all_close[(all_close["id"] == ticker) & (all_close["timestamp"] < test_date)]["Close"]
            if len(close_current_ser) > 0 and close_current_ser.iloc[-1] > 0 and pred_close > 0:
                close_current = float(close_current_ser.iloc[-1])
                pred_ret = np.log(pred_close / close_current)
            else:
                pred_ret = 0.0

            pred_store[(ticker, test_date)] = pred_ret

    except Exception as e:
        print(f"    WARNING step {step}: {e} -> zero pred")
        for ticker in tickers:
            pred_store[(ticker, test_date)] = 0.0

print(f"Forecast steps completed: {len(test_dates)}")

Running rolling inference...
  Step 1/876 | date=2022-07-01


KeyboardInterrupt: 

In [ ]:
# ========== ASSEMBLE RESULTS ==========
rows = []
for _, row in test_panel.iterrows():
    key      = (row["id"], row["timestamp"])
    pred_ret = pred_store.get(key, 0.0)
    rows.append({
        "Date"      : row["timestamp"],
        "ticker"    : row["id"],
        "target"    : row["target"],
        "regime_hmm": row["regime_hmm"],
        "pred"      : pred_ret,
    })

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "chronos_results.csv", index=False)
print(f"Saved: {OUT_DIR / 'chronos_results.csv'}")
print(results.head())

In [ ]:
# ========== SUMMARY METRICS ==========
summary_rows = []
for ticker in tickers:
    sub = results[results["ticker"] == ticker]
    m   = eval_metrics(sub["target"].values, sub["pred"].values)
    print(f"[{ticker}] MAE={m['MAE']:.6f} | RMSE={m['RMSE']:.6f} | DA={m['DA']:.3f} | IC={m['IC']:.4f}")
    summary_rows.append({"ticker": ticker, **m})

summary = pd.DataFrame(summary_rows)
overall = eval_metrics(results["target"].values, results["pred"].values)
summary.loc[len(summary)] = {"ticker": "OVERALL", **overall}
summary.to_csv(OUT_DIR / "chronos_summary.csv", index=False)

print("
" + "=" * 60)
print("Chronos-2 — OVERALL METRICS")
print("=" * 60)
print(summary.round(6).to_string(index=False))

In [ ]:
# ========== PER-REGIME BREAKDOWN ==========
print("
Per-regime breakdown:")
for regime in ["Bull", "Sideways", "Bear"]:
    mask = results["regime_hmm"] == regime
    if mask.sum() == 0:
        continue
    m = eval_metrics(results.loc[mask, "target"].values, results.loc[mask, "pred"].values)
    print(f"  {regime:10s}: MAE={m['MAE']:.6f} | RMSE={m['RMSE']:.6f} | DA={m['DA']:.3f} | IC={m['IC']:.4f} | n={mask.sum()}")

print(f"
Saved: {OUT_DIR / 'chronos_results.csv'}")
print(f"Saved: {OUT_DIR / 'chronos_summary.csv'}")